In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 05_facts/01_fact_encounters
# Purpose  : fact_encounters - Primary grain for the entire project.
#            Grain: One row = one visit (one encounter).
#
# Critical Technical Note (Fixing a Production Join Bug):
#    Attempting a Point-in-Time Join (matching start_datetime against __START_AT/__END_AT)
#    failed entirely, returning patient_key = NULL across all rows. Reason: __START_AT
#    is derived from _ingested_at (pipeline runtime timestamp, e.g., 2026), which is
#    a technical metadata timestamp rather than a business event timestamp. In contrast,
#    encounters.start_datetime contains historical business timestamps (1909–2020).
#    Consequently, the evaluation condition "visit date >= pipeline execution timestamp"
#    fails for all historical records.
#
#    Resolution: Since dim_patient currently maintains a single record version per patient
#    (static snapshot without true historical delta updates, as established during dim_patient design),
#    complex Point-in-Time logic yields no practical benefit here. We replaced it with a standard
#    Join filtered on is_current = true. If the pipeline transitions to a true incremental pattern
#    with actual temporal updates and a valid business timestamp source (e.g., "address_last_updated_at"),
#    Point-in-Time joins should be reintroduced using that business event time instead of _ingested_at.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.fact_encounters",
    comment="Fact table - one row per healthcare encounter",
    cluster_by=["patient_key", "start_date_key"],
)
def fact_encounters():
    encounters = dlt.read_stream("covid19_lakehouse.silver.encounters")

    patients = (
        spark.read.table("covid19_lakehouse.gold.dim_patient")
        .filter(F.col("is_current") == True)  # noqa: E712
        .select(F.col("patient_id"), F.col("patient_key"))
    )
    providers = spark.read.table("covid19_lakehouse.gold.dim_provider").select(
        F.col("provider_id"), F.col("provider_key")
    )
    organizations = spark.read.table("covid19_lakehouse.gold.dim_organization").select(
        F.col("organization_id"), F.col("organization_key")
    )
    payers = spark.read.table("covid19_lakehouse.gold.dim_payer").select(
        F.col("payer_id"), F.col("payer_key")
    )
    encounter_types = spark.read.table("covid19_lakehouse.gold.dim_encounter_type").select(
        F.col("encounter_code"), F.col("encounter_type_key")
    )
    reasons = spark.read.table("covid19_lakehouse.gold.dim_reason").select(
        F.col("reason_code"), F.col("reason_key")
    )

    df = (
        encounters
        .join(patients, on="patient_id", how="left")
        .join(providers, on="provider_id", how="left")
        .join(organizations, on="organization_id", how="left")
        .join(payers, on="payer_id", how="left")
        .join(encounter_types, on="encounter_code", how="left")
        .join(reasons, on="reason_code", how="left")
    )

    return df.select(
        F.col("encounter_id"),
        F.col("patient_key"),
        F.col("provider_key"),
        F.col("organization_key"),
        F.col("payer_key"),
        F.col("encounter_type_key"),
        F.coalesce(F.col("reason_key"), F.lit(-1)).alias("reason_key"),  # Unknown Member
        F.date_format(F.col("start_datetime"), "yyyyMMdd").cast("int").alias("start_date_key"),
        F.date_format(F.col("stop_datetime"), "yyyyMMdd").cast("int").alias("stop_date_key"),
        F.col("start_datetime"),
        F.col("stop_datetime"),
        (
            (F.unix_timestamp("stop_datetime") - F.unix_timestamp("start_datetime")) / 60
        ).cast("int").alias("duration_minutes"),
        F.col("base_encounter_cost"),
        F.col("total_claim_cost"),
        F.col("payer_coverage"),
        (F.col("total_claim_cost") - F.col("payer_coverage")).alias("patient_out_of_pocket"),
        F.coalesce(
            F.lower(F.col("reason_description")).contains("covid"), F.lit(False)
        ).alias("is_covid_related"),
    )